# consumer_couriers.ipynb

Real-Time Food Delivery Analytics — **COURIERS feed**

Consumes `group_10_couriers` from Azure Event Hubs, decodes Avro, flattens, and runs
three tiers of analytics:

| Tier | Query | Window |
|---|---|---|
| Basic        | Courier health KPIs per zone — tumbling 5-min | 5-min tumbling |
| Intermediate | Courier active sessions — session windows | — |
| Advanced     | Offline-mid-delivery anomaly detection | — |

## Prerequisites
1. `producer_couriers.ipynb` must have been run (at minimum the Test cell — more
   data = more interesting aggregates).
2. Colab Secret `CONSUMER_CONN_STR` must be set to the Consumer policy
   connection string for namespace `iesstsabbadbaa-grp-06-10`.

## How to run
Run cells top-to-bottom. Sections 1-7 are the standard scaffold (connection,
Spark setup, Avro decode, smoke test). Section 8 contains the analytics.
Each query in Section 8 has a **start cell** followed by a **peek cell** that
waits a few seconds and displays results. Run them in sequence.


## 1. Configuration

In [1]:
from google.colab import userdata

# Azure Event Hub configuration
event_hub_namespace = 'iesstsabbadbaa-grp-06-10'
eventhub_name       = 'group_10_couriers'

# Only the Consumer connection string is needed in this notebook.
consumer_eventhub_connection_str = userdata.get('CONSUMER_CONN_STR')

print(f"Consumer notebook configured for couriers feed: {eventhub_name}")


Consumer notebook configured for couriers feed: group_10_couriers


In [2]:
!pip install -q fastavro confluent-kafka


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 137.0 MB/s eta 0:00:00


## 2. Spark Setup

In [3]:
import os
import subprocess

# Fetch the latest Spark 4.x.x version
spark_version = subprocess.run(
    "curl -s https://downloads.apache.org/spark/ | grep -o 'spark-4\\.1\\+\\.[0-9]\\+' | sort -V | tail -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

spark_release = spark_version
hadoop_version = 'hadoop3'

os.environ['SPARK_RELEASE'] = spark_release
os.environ['HADOOP_VERSION'] = hadoop_version
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = f"/content/{spark_release}-bin-{hadoop_version}"

print(f"Spark release: {spark_release}")


Spark release: spark-4.1.1


In [4]:
# Install Java 21 and Spark 4.x.y
!apt-get update -qq
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/${SPARK_RELEASE}/${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz
!tar xf ${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz

!pip install -q findspark

import findspark
findspark.init()

import pyspark
print("PySpark version:", pyspark.__version__)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
PySpark version: 4.1.1


In [5]:
!echo "==== Java JDK: $JAVA_HOME ===="
!/usr/lib/jvm/java-21-openjdk-amd64/bin/java --version
!echo "==== Spark binaries ===="
!ls ${SPARK_RELEASE}-bin-${HADOOP_VERSION}/bin/ | head -5


==== Java JDK: /usr/lib/jvm/java-21-openjdk-amd64 ====
[0.010s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.010s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk 21.0.10 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)
==== Spark binaries ====
beeline
beeline.cmd
docker-image-tool.sh
find-spark-home
find-spark-home.cmd


## 3. Create Spark Session

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.avro.functions import from_avro

spark = SparkSession \
    .builder \
    .appName("StreamingAVROFromEventHub") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config('spark.jars.packages',
            'org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,'
            'org.apache.spark:spark-avro_2.13:4.1.1') \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark session ready.")


Spark session ready.


## 4. Avro Schema

In [7]:
# Avro schema as a Python string — MUST exactly match the producer's schema.
# Note: JSON uses lowercase null / false, not Python's None / False.
schema = """
{
    "type": "record",
    "name": "CourierEvent",
    "namespace": "com.fooddelivery.events.v1",
    "doc": "Periodic courier location and status ping.",
    "fields": [
        {"name": "event_id",    "type": "string"},
        {"name": "courier_id",  "type": "string"},
        {"name": "order_id",    "type": ["null", "string"], "default": null},
        {"name": "zone_id",     "type": "string"},
        {
            "name": "status",
            "type": {
                "type": "enum",
                "name": "CourierStatus",
                "symbols": [
                    "AVAILABLE", "ASSIGNED", "EN_ROUTE_PICKUP",
                    "EN_ROUTE_DELIVERY", "OFFLINE"
                ]
            }
        },
        {"name": "event_time",    "type": "string"},
        {"name": "ingestion_time","type": "string",  "default": ""},
        {"name": "latitude",      "type": "float"},
        {"name": "longitude",     "type": "float"},
        {"name": "speed_kmh",     "type": "float"},
        {"name": "battery_pct",   "type": "int"},
        {"name": "is_late_event", "type": "boolean","default": false},
        {"name": "is_duplicate",  "type": "boolean","default": false},
        {"name": "schema_version","type": "string", "default": "1.0"}
    ]
}
"""


## 5. Kafka (Event Hub) Consumer Config

In [8]:
# Kafka/Event Hub consumer configuration
kafkaConf = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    "kafka.sasl.mechanism":    "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config":  (
        f'org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" '
        f'password="{consumer_eventhub_connection_str}";'
    ),
    "subscribe":        eventhub_name,
    "startingOffsets":  "earliest",
    "groupIdPrefix":    "FoodDelivery_Couriers_Analytics",
    "enable.auto.commit":       "true",
    "auto.commit.interval.ms":  "5000",
}

print("Kafka config ready for topic:", eventhub_name)


Kafka config ready for topic: group_10_couriers


## 6. Read Stream + Decode Avro

In [9]:
# Read from Event Hub using the Kafka API
df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafkaConf) \
    .load()

df.printSchema()


root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [10]:
# Deserialize the Avro payload stored in Kafka's 'value' column
df = df.select(from_avro(df.value, schema).alias("courier"))
df.printSchema()


root
 |-- courier: struct (nullable = true)
 |    |-- event_id: string (nullable = false)
 |    |-- courier_id: string (nullable = false)
 |    |-- order_id: string (nullable = true)
 |    |-- zone_id: string (nullable = false)
 |    |-- status: string (nullable = false)
 |    |-- event_time: string (nullable = false)
 |    |-- ingestion_time: string (nullable = false)
 |    |-- latitude: float (nullable = false)
 |    |-- longitude: float (nullable = false)
 |    |-- speed_kmh: float (nullable = false)
 |    |-- battery_pct: integer (nullable = false)
 |    |-- is_late_event: boolean (nullable = false)
 |    |-- is_duplicate: boolean (nullable = false)
 |    |-- schema_version: string (nullable = false)



In [11]:
from pyspark.sql.functions import col, to_timestamp

df = df.select(
    col("courier.event_id"),
    col("courier.courier_id"),
    col("courier.order_id"),
    col("courier.zone_id"),
    col("courier.status"),
    col("courier.event_time"),
    col("courier.ingestion_time"),
    col("courier.latitude"),
    col("courier.longitude"),
    col("courier.speed_kmh"),
    col("courier.battery_pct"),
    col("courier.is_late_event"),
    col("courier.is_duplicate"),
    col("courier.schema_version"),
)

df = df.withColumn("event_time_ts", to_timestamp(col("event_time")))

df.printSchema()


root
 |-- event_id: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- ingestion_time: string (nullable = true)
 |-- latitude: float (nullable = true)
 |-- longitude: float (nullable = true)
 |-- speed_kmh: float (nullable = true)
 |-- battery_pct: integer (nullable = true)
 |-- is_late_event: boolean (nullable = true)
 |-- is_duplicate: boolean (nullable = true)
 |-- schema_version: string (nullable = true)
 |-- event_time_ts: timestamp (nullable = true)



## 7. Smoke Test — verify events are flowing

In [12]:
!mkdir -p checkpoint output checkpoint/q1 checkpoint/q2 checkpoint/q3


In [14]:
# Smoke test: stream everything into an in-memory table so we can peek at it.
import time

query_name = "all_couriers"

# If the query already exists from a previous run, skip re-creation
if query_name not in [q.name for q in spark.streams.active]:
    smoke_query = df.writeStream \
        .outputMode("append") \
        .format("memory") \
        .queryName(query_name) \
        .start()
    print(f"Streaming into in-memory table '{query_name}' — waiting 30s for events ...")
    time.sleep(30)
else:
    print(f"Stream '{query_name}' already running — just querying the table.")

print("\nSample events:")
spark.sql(f"SELECT * FROM {query_name} LIMIT 10").show(truncate=False)

print("\nTotal events received so far:")
spark.sql(f"SELECT count(*) AS n FROM {query_name}").show()


Stream 'all_couriers' already running — just querying the table.

Sample events:
+------------------------------------+------------+------------------------------------+------------+-----------------+------------------------+------------------------+---------+---------+---------+-----------+-------------+------------+--------------+-------------------+
|event_id                            |courier_id  |order_id                            |zone_id     |status           |event_time              |ingestion_time          |latitude |longitude|speed_kmh|battery_pct|is_late_event|is_duplicate|schema_version|event_time_ts      |
+------------------------------------+------------+------------------------------------+------------+-----------------+------------------------+------------------------+---------+---------+---------+-----------+-------------+------------+--------------+-------------------+
|8a86bddc-f630-43f0-bcb3-55e9e8a833f0|courier_0016|NULL                                |zone_nort

In [15]:
# List all active streaming queries
for q in spark.streams.active:
    print(f"Name: {q.name} | ID: {q.id} | Active: {q.isActive}")


Name: all_couriers | ID: 1c20025f-13f7-4406-b9b4-c72c2d0590a7 | Active: True


## 8. Analytics Queries

Three use cases at increasing difficulty.
Each block below *starts* a streaming query and then *peeks* at its output
after a short wait. The queries all write to in-memory tables so you can
re-query them at any time with standard SQL:

```python
spark.sql("SELECT * FROM <table_name> WHERE ...").show()
```

Query helpers:


In [16]:
# Helper: wait N seconds and display the current state of an aggregate table.
# Use after starting a streaming query to peek at its output.

import time

def peek(query_name: str, order_by: str = "window_start DESC", n: int = 15, wait: int = 30):
    """Wait a bit for the query to accumulate output, then show top rows."""
    print(f"Waiting {wait}s for '{query_name}' to accumulate output ...")
    time.sleep(wait)
    try:
        spark.sql(f"SELECT * FROM {query_name} ORDER BY {order_by} LIMIT {n}").show(truncate=False)
        spark.sql(f"SELECT count(*) AS rows_in_table FROM {query_name}").show()
    except Exception as e:
        print(f"[WARN] Could not query table yet: {e}")


In [17]:
def stop_query(name: str) -> None:
    """Stop a single streaming query by name — useful when iterating on a query."""
    for q in spark.streams.active:
        if q.name == name:
            print(f"Stopping {name} ...")
            q.stop()
            return
    print(f"No active query named {name}.")


### 8.1 USE CASE 1 (BASIC) — Courier health KPIs per zone — tumbling 5-min

In [24]:
# ─── USE CASE 1 (BASIC) ───────────────────────────────────────────────
# Courier health KPIs per zone, 5-minute tumbling windows.
# Uses approx_count_distinct (HyperLogLog) because exact countDistinct
# is not supported on streaming DataFrames.

from pyspark.sql.functions import (
    window, col, approx_count_distinct, avg, when, round as _round
)

courier_kpis = df \
    .withWatermark("event_time_ts", "10 minutes") \
    .groupBy(
        window(col("event_time_ts"), "5 minutes"),
        col("zone_id"),
    ) \
    .agg(
        approx_count_distinct("courier_id").alias("active_couriers"),
        approx_count_distinct(
            when(col("status") == "AVAILABLE", col("courier_id"))
        ).alias("available_couriers"),
        _round(avg("speed_kmh"), 1).alias("avg_speed_kmh"),
        _round(avg("battery_pct"), 1).alias("avg_battery_pct"),
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("zone_id"),
        col("active_couriers"),
        col("available_couriers"),
        col("avg_speed_kmh"),
        col("avg_battery_pct"),
    )

q1_courier_kpis = courier_kpis.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName("courier_kpis_per_zone_5min") \
    .option("checkpointLocation", "checkpoint/q1") \
    .start()

print("Started: courier_kpis_per_zone_5min")

Started: courier_kpis_per_zone_5min


In [32]:
peek("courier_kpis_per_zone_5min",
     order_by="window_start DESC, zone_id", n=20, wait=45)


Waiting 45s for 'courier_kpis_per_zone_5min' to accumulate output ...
+-------------------+-------------------+------------+---------------+------------------+-------------+---------------+
|window_start       |window_end         |zone_id     |active_couriers|available_couriers|avg_speed_kmh|avg_battery_pct|
+-------------------+-------------------+------------+---------------+------------------+-------------+---------------+
|2026-04-18 18:15:00|2026-04-18 18:20:00|zone_centre |15             |12                |17.6         |0.0            |
|2026-04-18 18:15:00|2026-04-18 18:20:00|zone_east   |11             |7                 |15.2         |0.0            |
|2026-04-18 18:15:00|2026-04-18 18:20:00|zone_north  |4              |3                 |17.2         |0.0            |
|2026-04-18 18:15:00|2026-04-18 18:20:00|zone_south  |7              |6                 |17.5         |0.0            |
|2026-04-18 18:15:00|2026-04-18 18:20:00|zone_suburbs|4              |4                 |2

### 8.2 USE CASE 2 (INTERMEDIATE) — Courier active sessions — session windows

In [33]:
# ─── USE CASE 2 (INTERMEDIATE) ────────────────────────────────────────
# Courier "active session" tracking using session windows.
#
# A session window is defined by a gap: if a courier doesn't ping for
# longer than the gap duration, Spark closes the current session and
# starts a new one on the next ping.
#
# With a 30-second ping interval and a 3-minute gap, any real gap in
# pings (e.g. courier goes OFFLINE, device reboots) will split into a
# separate session. This gives us session duration per courier — a
# natural "how long was this courier online continuously" metric.

from pyspark.sql.functions import session_window, col, min as _min, max as _max, count

courier_sessions = df \
    .withWatermark("event_time_ts", "10 minutes") \
    .groupBy(
        session_window(col("event_time_ts"), "3 minutes"),
        col("courier_id"),
    ) \
    .agg(
        count("*").alias("pings"),
        _min("event_time_ts").alias("session_start"),
        _max("event_time_ts").alias("session_end"),
    ) \
    .select(
        col("courier_id"),
        col("session_window.start").alias("window_start"),
        col("session_window.end").alias("window_end"),
        col("pings"),
        ((col("session_window.end").cast("long") -
          col("session_window.start").cast("long")) / 60.0).alias("session_minutes"),
    )

q2_sessions = courier_sessions.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("courier_sessions") \
    .option("checkpointLocation", "checkpoint/q2") \
    .start()

print("Started: courier_sessions")


Started: courier_sessions


In [34]:
peek("courier_sessions",
     order_by="window_start DESC, courier_id", n=20, wait=60)

# Summary — average session length and how many sessions per courier
print("\nSession summary per courier (top 10 by session count):")
spark.sql("""
    SELECT courier_id,
           count(*) AS num_sessions,
           round(avg(session_minutes), 2) AS avg_session_min,
           round(max(session_minutes), 2) AS longest_session_min
    FROM courier_sessions
    GROUP BY courier_id
    ORDER BY num_sessions DESC
    LIMIT 10
""").show(truncate=False)


Waiting 60s for 'courier_sessions' to accumulate output ...
+------------+-------------------+-------------------+-----+---------------+
|courier_id  |window_start       |window_end         |pings|session_minutes|
+------------+-------------------+-------------------+-----+---------------+
|courier_0027|2026-04-18 14:14:58|2026-04-18 14:17:58|1    |3.0            |
|courier_0031|2026-04-18 14:14:58|2026-04-18 14:17:58|2    |3.0            |
|courier_0036|2026-04-18 14:14:58|2026-04-18 14:17:58|1    |3.0            |
|courier_0024|2026-04-18 14:14:28|2026-04-18 14:17:28|1    |3.0            |
|courier_0020|2026-04-18 14:13:58|2026-04-18 14:16:58|2    |3.0            |
|courier_0021|2026-04-18 14:13:58|2026-04-18 14:16:58|1    |3.0            |
|courier_0042|2026-04-18 14:13:58|2026-04-18 14:16:58|1    |3.0            |
|courier_0007|2026-04-18 14:13:28|2026-04-18 14:16:28|1    |3.0            |
|courier_0023|2026-04-18 14:12:58|2026-04-18 14:15:58|1    |3.0            |
|courier_0029|20

### 8.3 USE CASE 3 (ADVANCED) — Offline-mid-delivery anomaly detection

In [39]:
# ─── USE CASE 3 (ADVANCED) ────────────────────────────────────────────
# Detect couriers going OFFLINE during peak delivery activity per zone.
#
# Since the generator clears order_id when setting status=OFFLINE, we
# detect the issue at the zone level: any 5-minute window where a zone
# has both elevated active-delivery activity AND OFFLINE events suggests
# a dispatch/connectivity problem affecting service.
#
# Heuristic: in a 10-min window, count OFFLINE events per zone. If a
# zone has more than 2 OFFLINE events in the window, flag it as a
# potential dispatch issue.

from pyspark.sql.functions import window, col, count, when, sum as _sum

offline_anomalies = df \
    .withWatermark("event_time_ts", "10 minutes") \
    .groupBy(
        window(col("event_time_ts"), "10 minutes"),
        col("zone_id"),
    ) \
    .agg(
        count(when(col("status") == "OFFLINE", True)).alias("offline_events"),
        count(when(col("status").isin("EN_ROUTE_PICKUP", "EN_ROUTE_DELIVERY"), True))
            .alias("active_deliveries"),
        count("*").alias("total_pings"),
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("zone_id"),
        col("offline_events"),
        col("active_deliveries"),
        col("total_pings"),
        when(col("offline_events") >= 3, True).otherwise(False).alias("has_dispatch_issue"),
    )

q3_offline_anomalies = offline_anomalies.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName("offline_mid_delivery_anomalies") \
    .option("checkpointLocation", "checkpoint/q3") \
    .start()

print("Started: offline_mid_delivery_anomalies (v2)")


Started: offline_mid_delivery_anomalies (v2)


In [44]:
peek("offline_mid_delivery_anomalies",
     order_by="window_start DESC, zone_id", n=20, wait=45)


Waiting 45s for 'offline_mid_delivery_anomalies' to accumulate output ...
+-------------------+-------------------+------------+--------------+-----------------+-----------+------------------+
|window_start       |window_end         |zone_id     |offline_events|active_deliveries|total_pings|has_dispatch_issue|
+-------------------+-------------------+------------+--------------+-----------------+-----------+------------------+
|2026-04-18 18:10:00|2026-04-18 18:20:00|zone_centre |9             |44               |236        |true              |
|2026-04-18 18:10:00|2026-04-18 18:20:00|zone_east   |7             |48               |173        |true              |
|2026-04-18 18:10:00|2026-04-18 18:20:00|zone_north  |0             |20               |75         |false             |
|2026-04-18 18:10:00|2026-04-18 18:20:00|zone_south  |1             |10               |105        |false             |
|2026-04-18 18:10:00|2026-04-18 18:20:00|zone_suburbs|0             |20               |64    

### 8.4 Exploration — run any SQL against the running tables

All four aggregate tables are now available as in-memory Spark tables.
Add cells here to explore the data interactively. Examples below.


In [ ]:
# Show all in-memory tables
spark.sql("SHOW TABLES").show(truncate=False)


In [ ]:
# Example: top revenue windows across all zones
# spark.sql('''
#     SELECT window_start, window_end, zone_id, revenue_eur
#     FROM orders_per_zone_5min
#     ORDER BY revenue_eur DESC
#     LIMIT 10
# ''').show(truncate=False)


## 9. Shutdown

In [ ]:
# Run this cell when you want to stop all streaming queries and the Spark session
if False:  # flip to True to actually stop
    for q in spark.streams.active:
        print(f"Stopping {q.name} ...")
        q.stop()
    spark.stop()
    print("Stopped.")
